In [ ]:
!pip install pyspark

In [ ]:
import os

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

Creates two directories for organizing dataset files.
data/raw stores original unprocessed data.
data/processed stores cleaned and transformed data.
exist_ok=True avoids error if folders already exist.
Helps maintain a structured ML project workflow.

In [ ]:
# Generate Sample Dataset

import csv
import random
import os

random.seed(42)

# Create folders
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

categories = [
    "Electronics",
    "Clothing",
    "Books",
    "Furniture",
    "Grocery",
    "Toys"
]

products = {
    "Electronics": ["Laptop", "Mouse", "Keyboard", "Headphones"],
    "Clothing": ["Shirt", "Jeans", "Jacket", "Shoes"],
    "Books": ["Python", "Spark", "SQL", "AI"],
    "Furniture": ["Chair", "Table", "Sofa", "Bed"],
    "Grocery": ["Rice", "Oil", "Sugar", "Milk"],
    "Toys": ["Car", "Puzzle", "Lego", "Doll"]
}

regions = ["North", "South", "East", "West", None]
status = ["DELIVERED", "PENDING", "CANCELLED", "RETURNED"]

rows = []

for i in range(1,111):

    category = random.choice(categories)

    row = {
        "order_id": f"ORD{1000+i}",
        "customer_id": f"CUST{random.randint(100,130)}",
        "product_name": random.choice(products[category]),
        "category": category,
        "price": round(random.uniform(100,5000),2),
        "quantity": random.randint(1,10),
        "region": random.choice(regions),
        "order_status": random.choice(status)
    }

    # Add some null values
    if random.random() < 0.08:
        row["price"] = None

    if random.random() < 0.05:
        row["product_name"] = None

    if random.random() < 0.06:
        row["quantity"] = None

    rows.append(row)

# Add duplicate rows
rows.extend(random.sample(rows,10))

random.shuffle(rows)

# Save CSV
with open("data/raw/orders.csv","w",newline="") as file:

    writer = csv.DictWriter(
        file,
        fieldnames=[
            "order_id",
            "customer_id",
            "product_name",
            "category",
            "price",
            "quantity",
            "region",
            "order_status"
        ]
    )

    writer.writeheader()
    writer.writerows(rows)

print("Dataset created successfully.")
print("Total rows:", len(rows))

Dataset created successfully.
Total rows: 120


Generates a synthetic e-commerce dataset using Python’s random module.
Creates 110+ records with categories, products, price, quantity, region, and status.
Introduces missing values and duplicate rows to simulate real-world data issues.
Stores raw data in data/raw/orders.csv for further processing.
Useful for practicing data cleaning, preprocessing, and analysis pipelines.




In [ ]:
# Create Spark Session

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("OrdersETL")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark Session Created Successfully!")

Spark Session Created Successfully!


Creates a SparkSession, which is the entry point for Spark applications. It sets the app name as OrdersETL and runs in local mode using all CPU cores. Shuffle partitions are reduced to 4 for efficient processing of small datasets. Log level is set to WARN to avoid unnecessary logs. This confirms Spark is initialized successfully.

In [ ]:
# Read Data & Transformations

from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType

# Read CSV
df = spark.read.csv(
    "data/raw/orders.csv",
    header=True,
    inferSchema=True
)

print("Original Data")
df.show(5)

print("Schema")
df.printSchema()


# Convert data types
df = (
    df.withColumn("price", F.col("price").cast(DoubleType()))
      .withColumn("quantity", F.col("quantity").cast(IntegerType()))
)

# Rename columns
df = (
    df.withColumnRenamed("product_name", "product")
      .withColumnRenamed("order_status", "status")
)

# Remove duplicates
df = df.dropDuplicates()

# Fill null values
df = df.fillna({
    "product": "UNKNOWN_PRODUCT",
    "price": 0.0,
    "quantity": 1,
    "region": "UNKNOWN_REGION"
})

# Add new columns
df = (
    df.withColumn(
        "total_price",
        F.round(F.col("price") * F.col("quantity"), 2)
    )
    .withColumn(
        "tax",
        F.round(F.col("total_price") * 0.05, 2)
    )
    .withColumn(
        "final_price",
        F.round(F.col("total_price") + F.col("tax"), 2)
    )
)

print("Cleaned Data")
df.show(5)

Original Data
+--------+-----------+------------+-----------+-------+--------+------+------------+
|order_id|customer_id|product_name|   category|  price|quantity|region|order_status|
+--------+-----------+------------+-----------+-------+--------+------+------------+
| ORD1078|    CUST125|      Jacket|   Clothing| 517.39|       4| North|    RETURNED|
| ORD1074|    CUST106|       Jeans|   Clothing|3719.16|       5|  NULL|   CANCELLED|
| ORD1017|    CUST124|         Oil|    Grocery| 729.12|       8|  NULL|     PENDING|
| ORD1077|    CUST108|  Headphones|Electronics|   NULL|    NULL|  West|    RETURNED|
| ORD1012|    CUST127|      Laptop|Electronics| 848.92|       3|  West|   DELIVERED|
+--------+-----------+------------+-----------+-------+--------+------+------------+
only showing top 5 rows
Schema
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double

Reads the CSV file into a Spark DataFrame and displays its structure using `show()` and `printSchema()`. It then casts `price` and `quantity` to proper numeric types for accurate calculations. Column names are renamed for simplicity, and duplicate rows are removed. Missing values are handled using default replacements to ensure data consistency. Finally, new columns like `total_price`, `tax`, and `final_price` are created using Spark transformations.


In [ ]:
# Filter, Aggregate and Save

from pyspark.sql import functions as F

# Filter delivered orders
filtered_df = df.filter(F.col("status") == "DELIVERED")

# Filter price greater than 0
filtered_df = filtered_df.filter(F.col("price") > 0)

print("Delivered Orders")
filtered_df.show(5)

# Revenue by Category
revenue_df = (
    filtered_df
    .groupBy("category")
    .agg(
        F.count("*").alias("total_orders"),
        F.round(F.sum("final_price"),2).alias("total_revenue"),
        F.round(F.avg("final_price"),2).alias("average_revenue")
    )
    .orderBy(F.desc("total_revenue"))
)

print("Revenue by Category")
revenue_df.show()

# Actions
print("Total Records :", filtered_df.count())
print("First Record :", filtered_df.first())
print("Top 3 Records :", filtered_df.take(3))

# Save as CSV
filtered_df.coalesce(1).write.mode("overwrite") \
.option("header",True) \
.csv("data/processed/csv")

# Save as Parquet
filtered_df.write.mode("overwrite") \
.parquet("data/processed/parquet")

print("CSV Saved Successfully")
print("Parquet Saved Successfully")

Delivered Orders
+--------+-----------+-------+--------+-------+--------+------+---------+-----------+-------+-----------+
|order_id|customer_id|product|category|  price|quantity|region|   status|total_price|    tax|final_price|
+--------+-----------+-------+--------+-------+--------+------+---------+-----------+-------+-----------+
| ORD1032|    CUST111| Jacket|Clothing| 872.96|       9|  East|DELIVERED|    7856.64| 392.83|    8249.47|
| ORD1110|    CUST129|     AI|   Books|4840.18|       5|  East|DELIVERED|    24200.9|1210.05|   25410.95|
| ORD1024|    CUST103|   Rice| Grocery|3295.51|       9| North|DELIVERED|   29659.59|1482.98|   31142.57|
| ORD1009|    CUST122| Puzzle|    Toys|3454.61|       1| South|DELIVERED|    3454.61| 172.73|    3627.34|
| ORD1004|    CUST104|  Spark|   Books|4790.34|       6| North|DELIVERED|   28742.04| 1437.1|   30179.14|
+--------+-----------+-------+--------+-------+--------+------+---------+-----------+-------+-----------+
only showing top 5 rows
Reven

Filters the dataset to keep only delivered orders with valid price values greater than 0. It then performs aggregation to calculate total orders, total revenue, and average revenue by category. Actions like `count`, `first`, and `take` are used to inspect the data. Finally, the cleaned results are saved in both CSV and Parquet formats for further analysis or downstream processing.


In [ ]:


from pyspark.sql import functions as F

# 1. Lazy Evaluation

print("Lazy Evaluation")

lazy_df = (
    df.filter(F.col("price") > 1000)
      .select("order_id", "product", "price")
)

print("No execution yet...")
print("Execution starts now:")
lazy_df.show(5)

# 2. Narrow Transformation

print("\nNarrow Transformation")

narrow_df = df.filter(F.col("category") == "Electronics")

narrow_df.explain()

# 3. Wide Transformation

print("\nWide Transformation")

wide_df = df.groupBy("category").sum("final_price")

wide_df.explain()

wide_df.show()

# 4. Catalyst Optimizer

print("\nCatalyst Optimizer")

filtered_df.filter(F.col("final_price") > 2000).explain(True)

# 5. Predicate Pushdown

print("\nPredicate Pushdown")

parquet_df = spark.read.parquet("data/processed/parquet")

parquet_df.filter(
    F.col("final_price") > 2000
).explain(True)

# 6. Column Pruning

print("\nColumn Pruning")

parquet_df.select(
    "order_id",
    "final_price"
).explain(True)

parquet_df.select(
    "order_id",
    "final_price"
).show(5)

Lazy Evaluation
No execution yet...
Execution starts now:
+--------+-------+-------+
|order_id|product|  price|
+--------+-------+-------+
| ORD1098|  Shoes|3929.47|
| ORD1011|   Milk|4499.33|
| ORD1067|    Bed|4807.86|
| ORD1110|     AI|4840.18|
| ORD1081|    Bed|1527.17|
+--------+-------+-------+
only showing top 5 rows

Narrow Transformation
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [order_id#17, customer_id#18, product#63, category#20, price#64, quantity#65, region#66, status#62, total_price#67, tax#68, round((total_price#67 + tax#68), 2) AS final_price#69]
   +- Project [order_id#17, customer_id#18, product#63, category#20, price#64, quantity#65, region#66, status#62, total_price#67, round((total_price#67 * 0.05), 2) AS tax#68]
      +- Project [order_id#17, customer_id#18, product#63, category#20, price#64, quantity#65, region#66, status#62, round((price#64 * cast(quantity#65 as double)), 2) AS total_price#67]
         +- HashAggregate(keys=[customer_id#

Shows how Spark uses lazy evaluation where transformations are executed only when an action like `show()` is called. It demonstrates narrow transformations (filter) and wide transformations (groupBy) using `explain()`. The Catalyst Optimizer improves query execution by optimizing logical plans. Predicate pushdown and column pruning are shown using Parquet to reduce unnecessary data scanning. Overall, it explains how Spark optimizes performance internally.
